In [12]:
from src import llm
from langchain.agents import create_agent

question = """
Диагностика экономического и пространственного состояния г. Тюмень:
- социально-экономический анализ (демография, производительность труда, структура и динамика ВГП, рынок
- труда, рынок жилья и коммерческой недвижимости, бюджетная обеспеченность);
- пространственный анализ (функциональная организация территории, структура землевладения, природноэкологический каркас, архитектурно-градостроительная среда);
- транспортный анализ (городской и внешний пассажирский и грузовой транспорт, парковки, пешеходные зоны);
- анализ инженерной инфраструктуры (водоснабжение и водоотведение, энергоснабжение, теплоснабжение).
"""

In [13]:
from pydantic import BaseModel, Field
    
class Role(BaseModel):
    name : str = Field(description='Название роли')
    description : str = Field(description='Описание роли')

class GeneratorResponse(BaseModel):
    roles : list[Role] = Field(min_length=1, description='Список экспертных ролей')

generator_agent = create_agent(model=llm, system_prompt="""
Вам дан вопрос. Предоставьте мне список из 1-{k_roles} экспертных ролей, которые наиболее полезны для решения вопроса.
Вопрос:
{question}
""".format(question=question, k_roles=3), response_format=GeneratorResponse)

roles = generator_agent.invoke({'messages':[]})['structured_response'].roles

In [14]:
roles

[Role(name='Градостроительный экономист‑аналитик', description='Эксперт, объединяющий экономический и пространственный анализ: оценивает демографию, структуру ВГП, рынок труда и недвижимости, а также функциональную организацию территории, землевладение и градостроительные процессы.'),
 Role(name='Транспортный и инфраструктурный планировщик', description='Специалист по оценке городской и внешней транспортной системы, парковок, пешеходных зон, а также инженерных сетей (водоснабжение, энергоснабжение, теплоснабжение) и их влияния на экономическое развитие.'),
 Role(name='Экологический и земельный аналитик', description='Эксперт, анализирующий природно‑экологический каркас, структуру землевладения, экологические ограничения и их взаимодействие с градостроительной средой и экономическими процессами.')]

In [18]:
_agent_prompt = """
Вы выдающийся специалист:
{role_name}
Описание:
{role_description}

Вы сотрудничаете с другими агентами для решения задачи. Существует общий документ, который вы можете читать и предлагать изменения или дополнения.

Вопрос:
{question}
"""

agents = []

from src.tools import ddgs_tool

for role in roles:
    agent = create_agent(model=llm, system_prompt=_agent_prompt.format(role_name=role.name, role_description=role.description, question=question))
    agents.append(agent)

In [ ]:
from langchain.messages import SystemMessage

content = ''

results = []

for i in range(1):
    for a in agents:
        response = a.invoke({'messages':SystemMessage(f"Содержание документа:\n{content}")})
        results.append(response['messages'][-1])

In [ ]:
_summarizer_prompt = """
Основываясь на сообщениях других агентов внеси изменения в исходный документ:
---
Исходный документ:
{document}
---
Сообщения агентов:
{messages}
"""

llm.invoke(SystemMessage(_summarizer_prompt.format(document=content, messages=results)))